1. Veri setini yükleme
2. MLP yapısını tanımlama
3. Model parametrelerini başlatma
4. İleri yayılım (forward propagation)
5. Maliyet (cost) hesaplama
6. Geri yayılım (Backpropagation)
7. Parametre güncelleme Update Propagation
8. Tüm adımların entegrasyonu

## Veri setini yükleme

In [47]:
import pandas as pd

In [48]:
df = pd.read_csv("drug_consumption_balanced.csv")

In [49]:
df['Cannabis'] = df['Cannabis'].apply(lambda x: 0 if x == 'CL0' else 1)

df = df.drop(columns= ['ID'])
cols = [c for c in df.columns if c != 'Cannabis'] + ['Cannabis']
df = df[cols]


In [50]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 826 entries, 0 to 825
Data columns (total 13 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Age        826 non-null    float64
 1   Gender     826 non-null    float64
 2   Education  826 non-null    float64
 3   Country    826 non-null    float64
 4   Ethnicity  826 non-null    float64
 5   Nscore     826 non-null    float64
 6   Escore     826 non-null    float64
 7   Oscore     826 non-null    float64
 8   Ascore     826 non-null    float64
 9   Cscore     826 non-null    float64
 10  Impulsive  826 non-null    float64
 11  SS         826 non-null    float64
 12  Cannabis   826 non-null    int64  
dtypes: float64(12), int64(1)
memory usage: 84.0 KB


In [51]:
df.head(50)

,Age,Gender,Education,Country,Ethnicity,Nscore,Escore,Oscore,Ascore,Cscore,Impulsive,SS,Cannabis
0,-0.95197,0.48246,1.16365,0.96082,-0.31685,-0.24649,0.47617,0.88309,-0.15487,1.81175,0.52975,0.40148,1
1,0.49788,0.48246,-0.05921,0.21128,-0.31685,0.04257,0.47617,-0.84732,-0.76096,1.13407,0.52975,0.76540,1
2,-0.07854,-0.48246,-0.61113,-0.57009,-0.31685,-0.34799,0.63779,1.88511,-0.60633,0.25953,0.19268,0.76540,1
3,0.49788,0.48246,1.98437,0.96082,-0.31685,0.52135,-0.30033,1.65653,0.28783,1.13407,-1.37983,-2.07848,1
4,-0.95197,-0.48246,-0.61113,-0.57009,-0.31685,-0.24649,0.16767,-0.97631,0.59042,0.93949,0.88113,0.40148,1
5,-0.07854,-0.48246,-0.05921,-0.57009,-0.31685,1.23461,-0.57545,0.88309,-0.01729,-0.89891,0.88113,0.76540,1
6,0.49788,0.48246,1.16365,0.96082,-0.31685,-0.58016,0.47617,-0.58331,-0.30172,0.93949,-0.21712,-0.84637,1
7,-0.07854,0.48246,1.98437,-0.57009,-0.31685,-1.05308,0.80523,-0.31776,-0.15487,0.93949,-0.21712,-0.52593,1
8,-0.07854,0.48246,-0.61113,-0.57009,-0.31685,-0.24649,-0.43999,-1.55521,0.13136,0.58489,-1.37983,-1.54858,1
9,-0.07854,0.48246,1.98437,0.96082,-0.31685,-1.05308,1.45421,-0.84732,0.94156,0.58489,-0.71126,-0.52593,1


In [52]:
df.describe()

,Age,Gender,Education,Country,Ethnicity,Nscore,Escore,Oscore,Ascore,Cscore,Impulsive,SS,Cannabis
count,826.000000,826.000000,826.000000,826.000000,826.000000,826.000000,826.000000,826.000000,826.000000,826.000000,826.000000,826.000000,826.0
mean,0.168080,0.085277,0.057608,0.508167,-0.328278,-0.060029,0.027327,-0.197196,0.110081,0.194614,-0.186178,-0.258899,1.0
std,0.909124,0.475151,0.984273,0.659379,0.173798,0.985466,0.978178,0.973696,0.969876,0.972190,0.938166,0.972517,0.0
min,-0.951970,-0.482460,-2.435910,-0.570090,-1.107020,-2.756960,-2.728270,-3.273930,-3.005370,-2.728270,-2.555240,-2.078480,1.0
25%,-0.951970,-0.482460,-0.611130,-0.285190,-0.316850,-0.763195,-0.575450,-0.847320,-0.606330,-0.405810,-0.711260,-0.846370,1.0
50%,-0.078540,0.482460,-0.059210,0.960820,-0.316850,-0.051880,0.003320,-0.177790,0.131360,0.259530,-0.217120,-0.215750,1.0
75%,1.094490,0.482460,0.454680,0.960820,-0.316850,0.629670,0.637790,0.445850,0.760960,0.939490,0.529750,0.401480,1.0
max,2.591710,0.482460,1.984370,0.960820,0.126000,3.273930,2.859500,2.449040,2.756960,3.464360,2.901610,1.921730,1.0


In [53]:
df.isnull().sum()

,0
Age,0
Gender,0
Education,0
Country,0
Ethnicity,0
Nscore,0
Escore,0
Oscore,0
Ascore,0
Cscore,0


In [54]:
df = df.sample(frac = 1, random_state = 42).reset_index(drop= True)

In [55]:
df.head(50)

,Age,Gender,Education,Country,Ethnicity,Nscore,Escore,Oscore,Ascore,Cscore,Impulsive,SS,Cannabis
0,-0.07854,0.48246,1.16365,0.96082,-0.22166,0.31287,0.63779,-0.58331,0.13136,-0.14277,0.19268,-0.21575,1
1,-0.95197,-0.48246,-0.61113,-0.57009,0.12600,-0.46725,2.12700,1.88511,-0.60633,-1.25773,1.86203,1.92173,1
2,-0.95197,-0.48246,-0.61113,-0.28519,-0.31685,-1.32828,-0.30033,0.88309,1.11406,-1.01450,-0.71126,-0.21575,1
3,1.09449,0.48246,1.16365,0.96082,-0.31685,-1.05308,1.74091,0.58331,0.43852,2.04506,-0.21712,-0.52593,1
4,-0.95197,0.48246,-0.61113,-0.57009,-0.31685,0.62967,0.63779,0.44585,0.13136,-2.57309,1.86203,1.22470,1
5,2.59171,-0.48246,0.45468,0.96082,-0.31685,-1.86962,-0.15487,-0.58331,2.46262,0.41594,-1.37983,-2.07848,1
6,-0.95197,-0.48246,-0.05921,0.96082,-0.31685,-0.92104,-0.30033,-0.71727,0.59042,0.41594,-0.71126,-1.54858,1
7,-0.95197,-0.48246,-1.43719,-0.09765,-0.31685,-0.34799,0.80523,0.72330,0.59042,-0.89891,1.86203,1.22470,1
8,1.09449,0.48246,0.45468,0.96082,-0.31685,0.62967,-0.57545,0.72330,-0.15487,0.41594,0.52975,-0.84637,1
9,1.09449,-0.48246,0.45468,0.96082,-0.31685,1.02119,-1.50796,-0.71727,0.28783,-0.89891,-0.21712,0.76540,1


In [56]:
X, y = df.iloc[:, :-1], df.iloc[:, -1]

In [57]:
print("X type is " + str(type(X)))
print("X shape is " + str(X.shape))
print("y type is " + str(type(y)))
print("y shape is " + str(y.shape))

X type is <class 'pandas.core.frame.DataFrame'>
X shape is (826, 12)
y type is <class 'pandas.core.series.Series'>
y shape is (826,)


In [58]:
X = X.to_numpy()


In [59]:
y = y.to_numpy().reshape(-1, 1)

In [60]:
print("X type is " + str(type(X)))
print("X shape is " + str(X.shape))
print("y type is " + str(type(y)))
print("y shape is " + str(y.shape))

X type is <class 'numpy.ndarray'>
X shape is (826, 12)
y type is <class 'numpy.ndarray'>
y shape is (826, 1)


In [61]:
from sklearn.model_selection import train_test_split

In [62]:
train_df = pd.read_csv("drug_train.csv")
test_df = pd.read_csv("drug_test.csv")

X_train = train_df.drop(columns=['Cannabis']).to_numpy()
y_train = train_df['Cannabis'].to_numpy().reshape(-1, 1)

X_test = test_df.drop(columns=['Cannabis']).to_numpy()
y_test = test_df['Cannabis'].to_numpy().reshape(-1, 1)

In [63]:
import numpy as np

print("Train class distribution:")
unique, counts = np.unique(y_train, return_counts=True)
print(dict(zip(unique, counts)))

print("Test class distribution:")
unique, counts = np.unique(y_test, return_counts=True)
print(dict(zip(unique, counts)))

Train class distribution:
{np.int64(0): np.int64(330), np.int64(1): np.int64(330)}
Test class distribution:
{np.int64(0): np.int64(83), np.int64(1): np.int64(83)}


In [64]:
print("X train shape is " + str(X_train.shape))
print("X test shape is " + str(X_test.shape))
print("y train shape is " + str(y_train.shape))
print("y test shape is " + str(y_test.shape))

X train shape is (660, 12)
X test shape is (166, 12)
y train shape is (660, 1)
y test shape is (166, 1)


## Model Mimarisi (Define Network Structure)

Input layer size is X.shape[1]
Hidden layer size is 5.
Output layer size is 1.

Örneğin; giriş katmanı boyutu X.shape[1], gizli katman nöron sayısı 5 ve çıkış katmanı 1 olabilir.

In [65]:
print("X.shape is " + str(X.shape))
print("X.shape[1] is " + str(X.shape[1]))

X.shape is (826, 12)
X.shape[1] is 12


## Modeli Başlatmak (Initialize Model Parameters)

In [66]:
def initialize_parameters(n_x, n_h, n_y = 1):
    np.random.seed(42)
    W1 = np.random.randn(n_h, n_x) * 0.01
    b1 = np.zeros((n_h, 1))
    W2 = np.random.randn(n_y, n_h) * 0.01
    b2 = np.zeros((n_y, 1))

    parameters = {
        "W1" : W1,
        "b1" : b1,
        "W2" : W2,
        "b2" : b2
    }
    return parameters

In [67]:
test_parameters = initialize_parameters(X.shape[1], 5, 1)

In [68]:
print("W1 = " + str(test_parameters["W1"]))
print("b1 = " + str(test_parameters["b1"]))
print("W2 = " + str(test_parameters["W2"]))
print("b2 = " + str(test_parameters["b2"]))

W1 = [[ 0.00496714 -0.00138264  0.00647689  0.0152303  -0.00234153 -0.00234137
   0.01579213  0.00767435 -0.00469474  0.0054256  -0.00463418 -0.0046573 ]
 [ 0.00241962 -0.0191328  -0.01724918 -0.00562288 -0.01012831  0.00314247
  -0.00908024 -0.01412304  0.01465649 -0.00225776  0.00067528 -0.01424748]
 [-0.00544383  0.00110923 -0.01150994  0.00375698 -0.00600639 -0.00291694
  -0.00601707  0.01852278 -0.00013497 -0.01057711  0.00822545 -0.01220844]
 [ 0.00208864 -0.0195967  -0.01328186  0.00196861  0.00738467  0.00171368
  -0.00115648 -0.00301104 -0.01478522 -0.00719844 -0.00460639  0.01057122]
 [ 0.00343618 -0.0176304   0.00324084 -0.00385082 -0.00676922  0.00611676
   0.01031     0.0093128  -0.00839218 -0.00309212  0.00331263  0.00975545]]
b1 = [[0.]
 [0.]
 [0.]
 [0.]
 [0.]]
W2 = [[-0.00479174 -0.00185659 -0.01106335 -0.01196207  0.00812526]]
b2 = [[0.]]


In [69]:
print("W1 = " + str(test_parameters["W1"].shape))
print("b1 = " + str(test_parameters["b1"].shape))
print("W2 = " + str(test_parameters["W2"].shape))
print("b2 = " + str(test_parameters["b2"].shape))
print("X_train = " + str(X_train.shape))

W1 = (5, 12)
b1 = (5, 1)
W2 = (1, 5)
b2 = (1, 1)
X_train = (660, 12)


## Forward Propagation

In [70]:
def forward_propagation(X, parameters):
    W1 = parameters["W1"]
    b1 = parameters["b1"]
    W2 = parameters["W2"]
    b2 = parameters["b2"]

    # gizli katman
    Z1 = np.dot(W1, X.T) + b1
    A1 = np.tanh(Z1)

    # çıkış katmanı
    Z2 = np.dot(W2, A1) + b2
    A2 = 1/ (1 + np.exp(-Z2))

    cache = {"Z1": Z1, "A1": A1, "Z2":Z2, "A2":A2}
    return A2, cache


In [71]:
def sigmoid(Z):
    return 1 / (1 + np.exp(-1 * Z))

In [72]:
test_A2, test_cache = forward_propagation(X_train, test_parameters)

In [73]:
print("A2 test:" + str(test_A2))
print("A2 test shape:" + str(test_A2.shape))

A2 test:[[0.4999994  0.4998063  0.50006475 0.50004023 0.49977039 0.49986767
  0.49978868 0.49990372 0.50006563 0.49973509 0.49999007 0.50003005
  0.5000181  0.50006597 0.49972793 0.50008532 0.50003179 0.49997648
  0.50009873 0.49994194 0.50005925 0.49994302 0.50006961 0.49980998
  0.50005478 0.4999435  0.50009845 0.49993997 0.50000087 0.49999897
  0.50013391 0.49995456 0.50010664 0.50012255 0.50000678 0.50011701
  0.50007812 0.49998519 0.50010599 0.50008712 0.50003405 0.50001519
  0.49996957 0.49994914 0.50013311 0.50004358 0.49999031 0.49997014
  0.50006223 0.49995231 0.50007382 0.49993929 0.50010635 0.5000238
  0.49998492 0.50013956 0.49984515 0.50008432 0.49989184 0.50008375
  0.50000799 0.49988905 0.50002982 0.50004425 0.49981808 0.49988721
  0.49997508 0.50015704 0.50006147 0.50003355 0.50015446 0.50004146
  0.50013044 0.49997655 0.5000226  0.50001012 0.49987755 0.50009493
  0.49999002 0.50004169 0.49996188 0.49980846 0.49997633 0.49992674
  0.49998315 0.50002319 0.49984301 0.5000

In [74]:
print("Z1 is:" + str(test_cache["Z1"]))
print("A1 is:" + str(test_cache["A1"]))
print("Z2 is:" + str(test_cache["Z2"]))
print("A2 is:" + str(test_cache["A2"]))

Z1 is:[[ 0.00647269 -0.0380391   0.02773644 ...  0.03057421 -0.00943798
   0.00230527]
 [ 0.03166208  0.06593302  0.01329956 ... -0.04510584  0.02984269
  -0.00797343]
 [-0.00439729  0.02695114 -0.03486895 ...  0.02394851 -0.00904847
  -0.00028134]
 [ 0.00260895  0.04612559 -0.02743478 ...  0.05543067  0.02163915
  -0.0297551 ]
 [ 0.00860622  0.00180842 -0.03658719 ...  0.04011146 -0.02158648
  -0.018871  ]]
A1 is:[[ 0.0064726  -0.03802076  0.02772933 ...  0.03056469 -0.0094377
   0.00230527]
 [ 0.0316515   0.06583764  0.01329877 ... -0.04507527  0.02983383
  -0.00797326]
 [-0.00439727  0.02694462 -0.03485482 ...  0.02394393 -0.00904822
  -0.00028134]
 [ 0.00260895  0.04609291 -0.0274279  ...  0.05537397  0.02163578
  -0.02974632]
 [ 0.00860601  0.00180841 -0.03657088 ...  0.04008996 -0.02158313
  -0.01886876]]
Z2 is:[[-2.41275527e-06 -7.74818110e-04  2.58995418e-04  1.60935683e-04
  -9.18422468e-04 -5.29323371e-04 -8.45283464e-04 -3.85119331e-04
   2.62533692e-04 -1.05965301e-03 -3.97

## Compute Cost

In [75]:
print("Y shape is " +str(y_train.shape))

Y shape is (660, 1)


In [76]:
def compute_cost(A2, Y):
    m = A2.shape[1]
    cost = - (np.dot(np.log(A2), Y) + np.dot(np.log(1 - A2), (1 - Y))) / m
    cost = float(np.squeeze(cost))
    return cost

In [77]:
test_cost = compute_cost(test_A2, y_train)

In [78]:
print("test_cost is " + str(test_cost))
print("test_cost type is " + str(type(test_cost)))

test_cost is 0.693171374234361
test_cost type is <class 'float'>


## Backpropagation

In [79]:
print("A1 shape" + str(test_cache["A1"].shape))
print("A2 Shape" + str(test_A2.shape))
print("A2 Shape" + str(test_cache["A2"].shape))
print("W1 shape" + str(test_parameters["W1"].shape))
print("W2 shape" + str(test_parameters["W2"].shape))
print("y train shape" + str(y_train.shape))
print("X_train shape " + str(X_train.shape))

A1 shape(5, 660)
A2 Shape(1, 660)
A2 Shape(1, 660)
W1 shape(5, 12)
W2 shape(1, 5)
y train shape(660, 1)
X_train shape (660, 12)


In [80]:
def backpropagation(X, Y, cache, parameters):
    m = X.shape[0]

    W2 = parameters["W2"]
    A1 = cache["A1"]
    A2 = cache["A2"]

    # çıkış katmanı gradyanları
    dZ2 = A2 - Y.T
    dW2 = np.dot(dZ2, A1.T) / m
    db2 = np.sum(dZ2, axis = 1, keepdims = True) / m

    # gizli katman gradyanları
    dZ1 = np.dot(W2.T, dZ2) * (1-A1**2)
    dW1 = np.dot(dZ1, X) / m
    db1 = np.sum(dZ1, axis=1, keepdims = True) / m

    grads = {"dW1": dW1, "db1": db1, "dW2": dW2, "db2": db2}
    return grads

In [81]:
test_grads = backpropagation(X_train, y_train, test_cache, test_parameters)

In [82]:
print("dW1 shape is" + str(test_grads["dW1"].shape))
print("dW2 shape is" + str(test_grads["dW2"].shape))
print("db1 shape is" + str(test_grads["db1"].shape))
print("db2 shape is" + str(test_grads["db2"].shape))
print("b1 shape is" + str(test_parameters["b1"].shape))

dW1 shape is(5, 12)
dW2 shape is(1, 5)
db1 shape is(5, 1)
db2 shape is(1, 1)
b1 shape is(5, 1)


In [83]:
def update_parameters(parameters, grads, learning_rate = 0.01):
    W1 = parameters["W1"]
    b1 = parameters["b1"]
    W2 = parameters["W2"]
    b2 = parameters["b2"]

    dW1 = grads["dW1"]
    db1 = grads["db1"]
    dW2 = grads["dW2"]
    db2 = grads["db2"]

    W1 -= learning_rate * dW1
    b1 -= learning_rate * db1
    W2 -= learning_rate * dW2
    b2 -= learning_rate * db2

    paramaters = {
        "W1" : W1,
        "b1" : b1,
        "W2" : W2,
        "b2" : b2
    }

    return parameters

In [84]:
test_learned_parameters = update_parameters(test_parameters, test_grads)

In [85]:
test_learned_parameters["b2"].shape

(1, 1)

In [86]:
def nn_model(X, Y, n_x, n_h, n_y, n_steps = 100, print_cost = True):
    parameters = initialize_parameters(n_x, n_h, n_y)

    for i in range(0, n_steps):
        A2, cache = forward_propagation(X, parameters)
        cost = compute_cost(A2, Y)
        grads = backpropagation(X, Y, cache, parameters)
        parameters = update_parameters(parameters, grads)

        if print_cost and i % 10 == 0:
            print("cost %i %f" %(i,cost))

    return parameters


In [87]:
X_train.shape

(660, 12)

In [88]:
parameters = nn_model(X_train, y_train, X_train.shape[1], n_h=3, n_y=1, n_steps=500)

cost 0 0.693162
cost 10 0.693149
cost 20 0.693137
cost 30 0.693124
cost 40 0.693111
cost 50 0.693098
cost 60 0.693085
cost 70 0.693072
cost 80 0.693058
cost 90 0.693043
cost 100 0.693028
cost 110 0.693012
cost 120 0.692995
cost 130 0.692978
cost 140 0.692959
cost 150 0.692939
cost 160 0.692919
cost 170 0.692896
cost 180 0.692873
cost 190 0.692847
cost 200 0.692820
cost 210 0.692791
cost 220 0.692760
cost 230 0.692726
cost 240 0.692690
cost 250 0.692652
cost 260 0.692610
cost 270 0.692566
cost 280 0.692518
cost 290 0.692466
cost 300 0.692411
cost 310 0.692351
cost 320 0.692286
cost 330 0.692217
cost 340 0.692142
cost 350 0.692062
cost 360 0.691975
cost 370 0.691882
cost 380 0.691782
cost 390 0.691674
cost 400 0.691558
cost 410 0.691433
cost 420 0.691298
cost 430 0.691154
cost 440 0.690999
cost 450 0.690832
cost 460 0.690652
cost 470 0.690460
cost 480 0.690253
cost 490 0.690032


In [89]:
def predict(parameters, X):
    A2, cache = forward_propagation(X, parameters)
    predicts = A2 > 0.5
    return predicts

In [90]:
predicts = predict(parameters, X_test)

In [91]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
y_true = y_test.flatten()
y_pred = predicts.flatten()
acc = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, average='binary')
recall = recall_score(y_true, y_pred, average='binary')
f1 = f1_score(y_true, y_pred, average='binary')
conf_matrix = confusion_matrix(y_true, y_pred)
print(f"Accuracy: {acc:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")
print("\nConfusion Matrix:")
print(conf_matrix)
print("\nClassification Report:")
print(classification_report(y_true, y_pred))

Accuracy: 0.8193
Precision: 0.8841
Recall: 0.7349
F1 Score: 0.8026

Confusion Matrix:
[[75  8]
 [22 61]]

Classification Report:
              precision    recall  f1-score   support

           0       0.77      0.90      0.83        83
           1       0.88      0.73      0.80        83

    accuracy                           0.82       166
   macro avg       0.83      0.82      0.82       166
weighted avg       0.83      0.82      0.82       166



In [92]:
parameters_n_h = [i for i in range(3, 11)]
parameters_n_steps = [i for i in range(100, 1100, 100)]
for n_h in parameters_n_h:
    for n_step  in parameters_n_steps:
        parameters = nn_model(X_train, y_train, X_train.shape[1], n_h = n_h, n_y = 1, n_steps = n_step, print_cost = False)
        predicts = predict(parameters, X_test)
        acc = accuracy_score(y_test.flatten(), predicts.flatten())
        print(f"n_h: {n_h}, n_step: {n_step}, acc: {acc}")

n_h: 3, n_step: 100, acc: 0.6746987951807228
n_h: 3, n_step: 200, acc: 0.7590361445783133
n_h: 3, n_step: 300, acc: 0.7771084337349398
n_h: 3, n_step: 400, acc: 0.8072289156626506
n_h: 3, n_step: 500, acc: 0.8192771084337349
n_h: 3, n_step: 600, acc: 0.8012048192771084
n_h: 3, n_step: 700, acc: 0.8012048192771084
n_h: 3, n_step: 800, acc: 0.8012048192771084
n_h: 3, n_step: 900, acc: 0.8012048192771084
n_h: 3, n_step: 1000, acc: 0.8072289156626506
n_h: 4, n_step: 100, acc: 0.7289156626506024
n_h: 4, n_step: 200, acc: 0.7831325301204819
n_h: 4, n_step: 300, acc: 0.8072289156626506
n_h: 4, n_step: 400, acc: 0.7951807228915663
n_h: 4, n_step: 500, acc: 0.7891566265060241
n_h: 4, n_step: 600, acc: 0.7951807228915663
n_h: 4, n_step: 700, acc: 0.8012048192771084
n_h: 4, n_step: 800, acc: 0.8012048192771084
n_h: 4, n_step: 900, acc: 0.8012048192771084
n_h: 4, n_step: 1000, acc: 0.8012048192771084
n_h: 5, n_step: 100, acc: 0.7289156626506024
n_h: 5, n_step: 200, acc: 0.7710843373493976
n_h: 5, 